In [1]:
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from industrial_inventory_env import (
    IndustrialInventoryEnv,
    generate_student_config,
    public_config_summary,
)

np.set_printoptions(suppress=True)

In [2]:
ROLL_NUMBER = "DA25M622"

student_config = generate_student_config(ROLL_NUMBER)
config_summary = public_config_summary(student_config)

print("Assigned configuration generated successfully.")
for key, value in config_summary.items():
    print(f"{key}: {value}")

Assigned configuration generated successfully.
project_version: IITM-6002W-RL-Inventory-2026-v1
roll_number: DA25M622
variant_id: V022
config_fingerprint: 7d77fc79debf206a
demand_multiplier_profile: [1.0, 1.1, 0.9]
initial_inventory_profile: [110, 100, 90]
lead_time_delay_profile: [0.1, 0.0, 0.05]


In [3]:
env = IndustrialInventoryEnv(
    student_config=student_config,
    scenario_mode="random",
    domain_randomization=True,
)

observation, info = env.reset(seed=2026)

print("Action space:", env.action_space)
print("Observation space:", env.observation_space)
print("Variant:", info["variant_id"])
print("Scenario components:", info["episode_parameters"]["scenario_components"])
print("Episode demand multipliers:", info["episode_parameters"]["demand_multipliers"])
print("Episode initial inventory:", info["episode_parameters"]["initial_inventory"])
print("Episode delay probabilities:", info["episode_parameters"]["delay_probabilities"])

Action space: MultiDiscrete([11 11 11])
Observation space: Dict('arrival_pipeline': Box(0, 10000, (3, 4), int32), 'capacity_utilisation': Box(0.0, 1.0, (1,), float32), 'day': Box(0, 50, (1,), int32), 'demand_history': Box(0, 10000, (7, 3), int32), 'inventory': Box(0, 1000, (3,), int32))
Variant: V022
Scenario components: ['seasonal', 'trend']
Episode demand multipliers: [1.05, 1.05, 0.85]
Episode initial inventory: [110, 100, 90]
Episode delay probabilities: [0.08, 0.0, 0.05]


In [4]:
for key, value in observation.items():
    print(f"{key:22s} shape={value.shape}, dtype={value.dtype}")
    print(value)
    print()

inventory              shape=(3,), dtype=int32
[110 100  90]

arrival_pipeline       shape=(3, 4), dtype=int32
[[0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]]

demand_history         shape=(7, 3), dtype=int32
[[0 0 0]
 [0 0 0]
 [0 0 0]
 [0 0 0]
 [0 0 0]
 [0 0 0]
 [0 0 0]]

day                    shape=(1,), dtype=int32
[0]

capacity_utilisation   shape=(1,), dtype=float32
[0.655]



In [5]:
action_indices = env.quantities_to_action_indices(quantities=[40, 20, 0])    # convert (just 0.1 x)
order_quantities = env.action_indices_to_quantities(action_indices).tolist()  # convert back (just 10 x)
next_observation, reward, terminated, truncated, step_info = env.step(action_indices)

print("Reward:", reward)
print("Terminated:", terminated)
print("Truncated:", truncated)
print("Demand:", step_info["demand"])
print("Daily costs:", step_info["costs"])
print("Next inventory:", next_observation["inventory"])

Reward: -27.425
Terminated: False
Truncated: False
Demand: [34 21 21]
Daily costs: {'holding': 2462.5, 'stockout': 0.0, 'ordering': 280.0, 'discarding': 0.0, 'daily_total': 2742.5, 'episode_total': 2742.5}
Next inventory: [76 79 69]


In [ ]:
# TODO: train


In [ ]:
# eval policy
eval_env = IndustrialInventoryEnv(
    student_config,
    scenario_mode="random",
    domain_randomization=True,
)
observation, reset_info = eval_env.reset(seed=101)

records = []
while True:
    quantities = demonstration_policy(observation)
    action = eval_env.quantities_to_action_indices(quantities)
    observation, reward, terminated, truncated, info = eval_env.step(action)

    records.append(
        {
            "day": info["day"],
            "reward": reward,
            "daily_cost": info["costs"]["daily_total"],
            "episode_cost": info["costs"]["episode_total"],
            "inventory_p1": int(observation["inventory"][0]),
            "inventory_p2": int(observation["inventory"][1]),
            "inventory_p3": int(observation["inventory"][2]),
            "order_p1": int(quantities[0]),
            "order_p2": int(quantities[1]),
            "order_p3": int(quantities[2]),
        }
    )

    if terminated or truncated:
        break

one_episode_results = pd.DataFrame(records)
print("Episode days:", len(one_episode_results))
print("Total episode cost:", one_episode_results["daily_cost"].sum())
one_episode_results